# Minilink showcase

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/showcase_minilink.ipynb)

**Write the equations once. Simulate, analyze, control, plan, optimize, learn.**

The story of the [README](https://github.com/alx87grd/minilink#readme), run end to end on one pendulum: a closed loop, diagrams
and the classical tools, derivatives and batches, one problem for every planner, one score, a
learned law, tuning through the loop, and the same lines on a robot. It runs in about a minute.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. A closed loop

A plant from the catalog, a controller, and `@` to close the loop. The loop is a system like
the plant.

In [ ]:
from minilink import Impedance, Pendulum

plant = Pendulum()
plant.x0 = [2.0, 0.0]

loop = Impedance(Kp=100.0, Kd=20.0) @ plant
loop.compute_trajectory(tf=6.0)
loop.plot_trajectory()

In [ ]:
loop.animate()

## 2. Diagrams in code

`>>` connects in series, `+` places blocks side by side, `@` in feedback, and `% Δ` runs a
block on a clock of period `Δ`. Here the same law runs on a computer at 10 Hz: the command is
held between two ticks, and the plant stays continuous.

In [ ]:
from minilink import Step

reference = Step(final_value=[1.0], step_time=1.0)
digital = reference >> (Impedance(Kp=100.0, Kd=20.0) % 0.1) @ plant

digital.compute_trajectory(tf=6.0)
digital.plot_diagram()
digital.plot_trajectory(signals=("x", "plant:u"))

The classical tools read any system, a loop included: `linearize`, `transfer_function`,
`plot_bode`, `plot_root_locus`, `plot_pzmap`, `plot_nyquist`, `plot_step_response`.

In [ ]:
print(loop.transfer_function())
loop.plot_bode()
loop.plot_root_locus()

`lqr` designs a state feedback on the linear model at a point (`kalman` and `lqg` are its
duals for estimation). At the upright, the pendulum balances.

In [ ]:
import numpy as np
from minilink import lqr

up = np.array([np.pi, 0.0])
balance = lqr(plant, Q=np.eye(2), R=np.eye(1), x_bar=up) @ plant
balance.compute_trajectory(tf=4.0, x0=[np.pi - 0.5, 0.0])
balance.animate()

A model of your own is its `f` (and `h` when the sensor does not read the whole state); every
tool then takes it like a catalog plant.

In [ ]:
from minilink import DynamicSystem


class MyPendulum(DynamicSystem):
    def __init__(self):
        super().__init__(state_dim=2, input_dim=1, output_dim=2)

    def f(self, x, u, t=0.0, params=None):
        m, l, g = 1.0, 1.0, 9.81
        theta, dtheta = x
        torque = u[0]

        # m l² θ̈ = τ − m g l sin θ
        ddtheta = (torque - m * g * l * np.sin(theta)) / (m * l**2)

        return np.array([dtheta, ddtheta])


my_loop = Impedance(Kp=100.0, Kd=20.0) @ MyPendulum()
my_loop.compute_trajectory(tf=6.0)
my_loop.plot_trajectory()

The wiring carries the numbers as well as the signals: a diagram's parameters are one tree,
and a path such as `"plant.l"` names any number in any block. Sections 3, 5 and 7 act on this
tree.

In [ ]:
loop.params

## 3. One `f`: derivatives and batches

`f` is a pure function of `(x, u, t; p)`. Under JAX its Jacobians are exact, in the state, the
input or the parameters.

In [ ]:
x = np.array([1.0, 0.0])
A = plant.jacobian("f", "x", x)
dfdp = plant.jacobian("f", "params", x)

print("A =", A.round(2).tolist())
print("df/dm =", dfdp["m"].round(2), " df/dgravity =", dfdp["gravity"].round(2))

Compiled once with JAX, the same `f` runs a thousand pendulums, each with its own rod length, in
one call.

In [ ]:
import matplotlib.pyplot as plt

lengths = np.linspace(0.5, 2.0, 1000)
starts = np.full((1000, 2), [1.0, 0.0])
swings = plant.compile(backend="jax").rollout_batch(
    starts, n_steps=1000, dt=0.005, params=dict(plant.params, l=lengths)
)

t = 0.005 * np.arange(1001)
for k in range(0, 1000, 200):
    plt.plot(t, swings[k, :, 0], label=f"l = {lengths[k]:.2f} m")
plt.xlabel("t [s]")
plt.ylabel("theta [rad]")
plt.legend()
plt.show()

## 4. One problem, every planner

A `PlanningProblem` states the task and nothing about how to solve it: the system, a cost, the
allowed states and inputs, the start, the goal and the horizon. The cost
$J = \int (\tilde{x}^T Q \tilde{x} + u^T R u)\,dt + \tilde{x}(t_f)^T S\, \tilde{x}(t_f)$,
$\tilde{x} = x - x_{up}$, weighs the distance to the upright, the torque, and where the swing
ends. Here, swing the pendulum up with a motor too weak to lift it directly.

In [ ]:
from minilink import BallSet, PlanningProblem, QuadraticCost

problem = PlanningProblem(
    plant, x0=[0.0, 0.0], Xf=BallSet(up, 0.2), horizon=6.0,
    cost=QuadraticCost.from_system(plant, x_bar=up, S=100 * np.eye(2)),
    X=plant.state.box, U=plant.input_box,
)

The same problem goes to planners that share nothing but it. **Value iteration** solves the
Bellman equation on a grid and returns the cost-to-go from every state.

In [ ]:
from minilink import DynamicProgrammingPlanner

vi = DynamicProgrammingPlanner(problem, x_grid=(101, 101), u_grid=(11,), dt=0.05).solve()
vi.plot_cost_to_go()

**RRT** grows a tree of motions until a branch reaches the goal.

In [ ]:
from minilink import RRTPlanner

rrt_planner = RRTPlanner(problem, seed=0)
rrt = rrt_planner.solve()
rrt_planner.plot_tree(x_axis=0, y_axis=1);

**Trajectory optimization** transcribes the problem into a nonlinear program and solves it.
Each answer holds a controller that is a system: value iteration's is a feedback law and closes
the loop with `@`; RRT's and trajectory optimization's are open-loop plans. The three on one
phase plane:

In [ ]:
from minilink import TrajectoryOptimizationPlanner

opt = TrajectoryOptimizationPlanner(problem, n_steps=60).solve()

swing_up = vi.controller @ plant
vi_run = swing_up.compute_trajectory(tf=6.0, x0=[0.0, 0.0])

phase = plant.plot_phase_plane(traj=opt.trajectory, show=False)
phase.axes.plot(*vi_run.x, "C0--", label="value iteration")
phase.axes.plot(*rrt.trajectory.x, "C2", label="RRT")
phase.axes.legend(loc="lower right")
phase.show()

**MPC** is a controller built on a planner and run on a clock: every 0.1 s it solves a 2 s
look-ahead again from the measured state.

In [ ]:
from minilink.control import ModelPredictiveController

look_ahead = problem.replace(horizon=2.0, Xf=None)
mpc = ModelPredictiveController(TrajectoryOptimizationPlanner(look_ahead, n_steps=20, backend="jax")) % 0.1

plant.x0 = [0.0, 0.0]
mpc_run = (mpc @ plant).compute_trajectory(tf=6.0)
plant.animate(mpc_run.of("plant"))

## 5. One score

`compare` runs every answer on the same problem: the same model, cost, sets and draws. Here the
mass is uncertain by ±20 %, and each controller runs on the same twenty draws.

In [ ]:
from minilink import Uniform, compare

uncertain = problem.replace(distributions={"plant": {"m": Uniform(0.8, 1.2)}})
comparison = compare(uncertain, VI=vi, RRT=rrt, TrajOpt=opt, MPC=mpc, n_trials=20)
print(comparison)

In [ ]:
comparison.plot_simulated_trajectory()

The two plans reach the goal on the model they were computed for; once the mass changes they
miss it and leave the state or torque limits on most draws. The two feedback laws reach it on
most draws and stay within the limits.

## 6. Learn a law

Reinforcement learning is one more planner, and it returns a neural network that is, again, a
controller. It learns from rewards alone, so its cost is shaped for learning: $1 + \cos\theta$
is lowest at the upright from every angle, where the quadratic cost above is not. The starts
are drawn at random.

In [ ]:
import jax.numpy as jnp

from minilink import CostFunction, ReinforcementLearningPlanner


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        return 1.0 + jnp.cos(x[0]) + 0.01 * x[1] ** 2 + 0.01 * u[0] ** 2

    def h(self, x, t=0.0, params=None):
        return 0.0


random_start = PlanningProblem(
    plant,
    cost=SwingUpCost(),
    horizon=np.inf,
    x0=Uniform([-np.pi, -1.0], [np.pi, 1.0]),
    U=plant.input_box,
)
learner = ReinforcementLearningPlanner(
    random_start,
    dt=0.05,
    features=lambda x: jnp.array([jnp.cos(x[0]), jnp.sin(x[0]), 0.1 * x[1]]),
    hidden=(32, 32),
    n_rollouts=64,
    n_steps_per_update=32,
    batch_size=256,
    learning_rate=3e-3,
    gamma=0.97,
)
learned = learner.solve(n_timesteps=120_000)
learner.plot_learning_curve();

In [ ]:
neural_loop = learned.controller @ plant
neural_loop.compute_trajectory(tf=8.0, dt=0.01, x0=[0.05, 0.0])
neural_loop.animate()

The same score on its own problem: the learned law against the LQR of section 2, on the
problem's random starts. The LQR ignores the motor's limit, and the count of feasible runs shows
it.

In [ ]:
print(compare(random_start, RL=learned, LQR=balance.subsystems["controller"], dt=0.05, n_trials=100, episode_length=10.0))

## 7. Tune through the loop

Every optimization, a trajectory, a set of gains, a network's weights, is lowered to one
program, $\min_z J(z)$ s.t. $g(z) \ge 0,\ h(z) = 0$, and an `Optimizer` solves it. Here the
task is to bring the pendulum, released at 2 rad, to rest. $z$ is three paths of the parameter
tree, the two gains and the rod length, each within bounds; $J$ is the cost of the closed-loop
run, with its gradient through the whole simulation.

In [ ]:
from minilink import Optimizer

task = PlanningProblem(plant, x0=[2.0, 0.0], horizon=5.0, cost=QuadraticCost.from_system(plant, R=0.1 * np.eye(1)))
evaluator = task.compile(Impedance(Kp=1.0, Kd=1.0), backend="jax")
tuned = ["controller.Kp", "controller.Kd", "plant.l"]
program = evaluator.program(
    tuned,
    bounds={"controller.Kp": (0.0, 50.0), "controller.Kd": (0.0, 20.0), "plant.l": (0.5, 2.0)},
)
best = Optimizer(program).solve()
print(best)

The shortest rod wins: the cost puts no price on it. The tuned loop:

In [ ]:
evaluator.loop.plot_trajectory(evaluator.trajectory(evaluator.unpack(best.z, tuned)))

## 8. The same lines on a robot

Nothing above is special to the pendulum. A six-axis arm with its rigid-body dynamics is the
same kind of system, and a task-space impedance law moves its hand to a target with the same
`>>` and `@`.

In [ ]:
from minilink import TaskImpedance, UR5Manipulator

robot = UR5Manipulator()
target = Step(initial_value=[-0.8, -0.2, 0.0], final_value=[-0.5, 0.0, 0.5], step_time=1.0)
arm_loop = target >> TaskImpedance(robot, gravity_comp=True) @ robot
arm_loop.animate(arm_loop.compute_trajectory(tf=5.0), renderer="meshcat")

On your own machine, `plant.game()` puts the keyboard or a gamepad on a system's input in real
time.

## Where to go next

- [`basics/`](basics/01_minilink.ipynb): model, connect, simulate, design a controller.
- [`advanced/`](advanced/01_discrete_and_hybrid.ipynb): sampled loops, optimal control, MPC, reinforcement learning, differentiable loops.
- [`core/`](core/01_systems.ipynb): the objects behind the tools, one by one.